In [ ]:
!pip install -qU langchain-google-genai langchain textstat

In [ ]:
import json
import hashlib
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import textstat

In [ ]:
# -------------------------------------------------------------------------
# Load source-of-truth assets: config.json + every prompt file declared
# in config.steps[*].prompt.messages
# -------------------------------------------------------------------------
# The canonical evaluator definition lives in evals/prompts/purpose.
# All consumers (DS notebook, Python SDK, TypeScript SDK) read these same files,
# so this loader is the pattern the SDK engineer will reproduce.

ASSETS_DIR = Path("./prompts/purpose")

config_path = ASSETS_DIR / "config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

# Load standalone schema files (config.json references them via $ref by path).
with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)
with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

# Load every prompt message declared in config (system, user, ...). Each
# message has {role, source_path, sha256}. We verify each file's sha256
# matches the declared hash -- drift tripwire #1, applied to every prompt
# regardless of role. CI should promote a mismatch to a hard failure.
PROMPT_MESSAGES = []  # list of (role, text) tuples, preserving config order
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    path = ASSETS_DIR / msg_spec["source_path"]
    text = path.read_text()
    actual_sha = hashlib.sha256(text.encode("utf-8")).hexdigest()
    declared_sha = msg_spec["sha256"]
    assert actual_sha == declared_sha, (
        f"prompt drift detected for role={role!r} ({msg_spec['source_path']}): "
        f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

print(
    f"Loaded {CONFIG['evaluator']['id']} "
    f"from {ASSETS_DIR.resolve()}"
)
print(f"  model:       {CONFIG['steps'][0]['model']['name']}")
print(f"  temperature: {CONFIG['steps'][0]['generation']['temperature']}")
print(f"  prompts:")
for msg_spec, (role, text) in zip(CONFIG["steps"][0]["prompt"]["messages"], PROMPT_MESSAGES):
    sha = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    print(f"    {role:>6}  {msg_spec['source_path']:<14} ({len(text):>5} chars, sha {sha})")

In [ ]:
# -------------------------------------------------------------------------
# FK score helper (declared as a preprocessing step in CONFIG['preprocessing'])
# -------------------------------------------------------------------------
_FK_PRE = next(p for p in CONFIG["preprocessing"] if p["id"] == "fk_score")
_FK_IMPL = _FK_PRE["implementation"]["python"]
_FK_LIB = _FK_IMPL["library"]
_FK_FN = _FK_IMPL["function"]
_FK_TRANSFORM = _FK_IMPL["post_transform"]

if _FK_LIB != "textstat":
    raise ValueError(f"unsupported fk library in config: {_FK_LIB!r}")


def calculate_fk_score(text) -> float:
    """Compute Flesch-Kincaid Grade Level per CONFIG['preprocessing']."""
    fn = getattr(textstat, _FK_FN)
    value = fn(text)
    if _FK_TRANSFORM["type"] == "round":
        value = round(value, _FK_TRANSFORM["precision"])
    else:
        raise ValueError(f"unsupported post_transform type: {_FK_TRANSFORM['type']!r}")
    return value


# -------------------------------------------------------------------------
# Evaluator function: model / prompt / parser config all read from CONFIG
# -------------------------------------------------------------------------
_STEP = CONFIG["steps"][0]  # single-step evaluator today. Extensible to multi-step evaluators.

def evaluate_text_complexity(text: str, grade_level: int):
    """
    Evaluate the Purpose-dimension complexity of a text using the canonical
    config in evals/prompts/purpose/config.json + system.txt + user.txt.

    Returns a dict with full I/O trace fields:
      - rendered_prompt:  the actual list of messages sent to the model
                          (input-side trace).
      - raw_output:       the AIMessage object returned by the LLM
                          (preserves response_metadata, usage_metadata).
      - raw_text:         just the string content of the AIMessage.
      - formatted_output: the parsed dict matching OUTPUT_SCHEMA.
      - usage:            token-usage metadata if the provider returned it.

    The LLM is invoked ONCE; include_raw=True returns both the raw AIMessage
    and the parsed output without a second call.
    """
    # 1. Structured output -- parser.kind == "structured_output" uses the model's
    #    native output enforcement. OUTPUT_SCHEMA is loaded from output_schema.json,
    #    the standalone source of truth. include_raw=True preserves the AIMessage
    #    for tracing alongside the parsed result.
    llm = ChatGoogleGenerativeAI(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)

    # 2. Prompt template -- every message's content was loaded from disk
    #    and verified against config in the loader cell. We just feed the
    #    (role, text) tuples straight into ChatPromptTemplate.
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES)

    try:
        # Step A: Calculate FK Score
        fk_score = calculate_fk_score(text)
        print(f"Calculated Flesch-Kincaid Score: {fk_score}")

        inputs = {"text": text, "grade_level": grade_level, "fk_score": fk_score}

        # Step B: Render the prompt up-front so we can return exactly what
        #         was sent to the model (input-side trace).
        rendered_messages = prompt_template.format_messages(**inputs)

        # Step C: Single LLM call -> raw AIMessage + parsed output dict.
        #         No second LLM call.
        raw = structured_llm.invoke(rendered_messages)

        if raw.get("parsing_error"):
            raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

        # Step D: Return the full trace dict.
        return {
            "rendered_prompt": [m.model_dump() for m in rendered_messages],
            "raw_output":       raw["raw"],
            "raw_text":         raw["raw"].content,
            "formatted_output": raw["parsed"],
            "usage":            getattr(raw["raw"], "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

In [ ]:
sample_text = """
"Well, then," said the teacher, "you may take your slate and go out behind the schoolhouse for half an hour. Think of something to write about, and write the word on your slate. Then try to tell what it is, what it is like, what it is good for, and what is done with it. That is the way to write a composition." Henry took his slate and went out. Just behind the schoolhouse was Mr. Finney's barn. Quite close to the barn was a garden. And in the garden, Henry saw a turnip. "Well, I know what that is," he said to himself; and he wrote the word turnip on his slate. Then he tried to tell what it was like, what it was good for, and what was done with it. Before the half hour was ended he had written a very neat composition on his slate. He then went into the house, and waited while the teacher read it. The teacher was surprised and pleased. He said, "Henry Longfellow, you have done very well. Today you may stand up before the school and read what you have written about the turnip."
"""

result = evaluate_text_complexity(
        text=sample_text,
        grade_level=4
    )

In [ ]:
import pprint as pp
# I/O trace breakdown -- this is what the SDK engineer will replicate in TS.
print("=" * 60)
print("RENDERED PROMPT (input sent to the LLM)")
print("=" * 60)
pp.pprint(result["rendered_prompt"])

print("\n" + "=" * 60)
print("RAW LLM TEXT (model's verbatim output)")
print("=" * 60)
print(result["raw_text"])

print("\n" + "=" * 60)
print("PARSED OUTPUT (output_schema)")
print("=" * 60)
pp.pprint(result["formatted_output"])

print("\n" + "=" * 60)
print("USAGE METADATA")
print("=" * 60)
pp.pprint(result["usage"])

In [ ]:
# -------------------------------------------------------------------------
# Sniff-test runner: load fixtures.json and check predictions against expected
# -------------------------------------------------------------------------
# Fixtures live next to config.json + system.txt + user.txt and follow the
# schema declared in CONFIG['fixtures']['schema']. Each case has:
#   - id, description (optional)
#   - input: {text, grade_level}                  -- runtime evaluator inputs
#   - expected: {complexity_level}                -- ground-truth label from rubric
#
# Note: the fixture key 'complexity_level' maps to the runtime output's
# 'complexity_score' field. We test that single value only -- the model's
# free-text 'reasoning' field is non-deterministic across runs.

fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["sniff_test_path"]
with open(fixtures_path) as f:
    fixtures = json.load(f)
print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

# Adjacency tolerance per CONFIG['fixtures']['tolerance'].
# Derive rubric order from OUTPUT_SCHEMA -- 'more_context_needed' excluded as it has no adjacency.
_RUBRIC_ORDER = [
    level for level in OUTPUT_SCHEMA["properties"]["complexity_score"]["enum"]
    if level != "more_context_needed"
]
_ALLOW_ADJ = bool(CONFIG["fixtures"]["tolerance"].get("allow_adjacent_levels", False))

def _score_outcome(predicted: str, expected: str):
    """Return ('exact' | 'adjacent' | 'fail', distance_or_None)."""
    if predicted == expected:
        return "exact", 0
    if _ALLOW_ADJ and predicted in _RUBRIC_ORDER and expected in _RUBRIC_ORDER:
        d = abs(_RUBRIC_ORDER.index(predicted) - _RUBRIC_ORDER.index(expected))
        if d == 1:
            return "adjacent", d
    return "fail", None

# Run each fixture, accumulate results
results = []
for fx in fixtures:
    expected = fx["expected"]["complexity_level"]
    out = evaluate_text_complexity(
        text=fx["input"]["text"],
        grade_level=fx["input"]["grade_level"],
    )
    if isinstance(out, str):  # error path
        results.append({"id": fx["id"], "status": "error", "predicted": None, "expected": expected, "error": out})
        continue
    predicted = out["formatted_output"]["complexity_score"]
    status, _ = _score_outcome(predicted, expected)
    results.append({
        "id": fx["id"], "status": status,
        "predicted": predicted, "expected": expected,
        "description": fx.get("description", ""),
    })

# Per-case report
print("\n" + "=" * 78)
print(f"{'ID':>5}  {'STATUS':<8}  {'PREDICTED':<22}  {'EXPECTED':<22}  DESCRIPTION")
print("=" * 78)
for r in results:
    icon = {"exact": "PASS", "adjacent": "PASS*", "fail": "FAIL", "error": "ERR"}[r["status"]]
    print(f"{r['id']:>5}  {icon:<8}  {(r['predicted'] or '-'):<22}  {r['expected']:<22}  {r.get('description','')[:25]}")

# Summary
n_total = len(results)
n_exact = sum(1 for r in results if r["status"] == "exact")
n_adj   = sum(1 for r in results if r["status"] == "adjacent")
n_fail  = sum(1 for r in results if r["status"] == "fail")
n_err   = sum(1 for r in results if r["status"] == "error")
print("=" * 78)
print(f"Summary: {n_exact} exact, {n_adj} adjacent (tolerated), {n_fail} fail, {n_err} error  --  total {n_total}")
if _ALLOW_ADJ:
    print("(Adjacency tolerance ON: predictions within +/-1 rubric step of the expected label count as PASS*.)")